# AQuA API — pre-promotion integration pass (v3 + v4)

Run this by hand against the **dev runner** before merging a `promote/*` branch into
`release`. It exists to catch what CI cannot: real HTTP end-to-end, and **parity between
v3 and v4 over the same database rows**. v4 has no clients yet, so a v4 regression is
easy to miss — this notebook is the thing that would notice.

Companion to `integration.ipynb` (v3 lifecycle) and `v4_smoke.ipynb` (v4 contract:
pagination envelope, `users/me` allowlist, versions write path). Neither of those
compares the two surfaces against each other; that is this notebook's job.

| § | What it covers |
|---|---|
| 0 | Setup — base URL, `.env`, admin token on **both** surfaces |
| 1 | v4 surface discovery — `GET /v4/`, operation count vs. the promotion PR |
| 2 | **v3/v4 parity** — same row, both surfaces, field for field |
| 3 | v4-only lifecycles — users & groups, reference lists, predictions, training, critique |
| 4 | Error contract — the `{"error": {...}}` envelope, `extra=forbid`, bad ids |
| 5 | v3 regression spot-checks — the `integration.ipynb` lifecycle, re-run |
| 6 | Teardown |
| 7 | Summary |

---

> ## ⚠️ Read before running
>
> **Dev and prod share one database.** Anything created here is a real row that prod's UI
> shows immediately. Every fixture is therefore named with a throwaway prefix and torn
> down in §6.
>
> **Three switches, all in §0.5:**
>
> | Switch | Default | What it gates |
> |---|---|---|
> | `RUN_WRITE_TESTS` | `False` | **Every write on both surfaces** — §2, §3.1, §3.5's PATCH, §5. With it off, a Run All is read-only. |
> | `RUN_TRAINING` | `False` | §3.4's `POST /v4/training-sessions`. Needs `RUN_WRITE_TESTS` **as well**. Real Modal GPU work, and a training job cannot be deleted until it is terminal — so this one can leave a row behind. |
> | `RUN_PREDICT_FANOUT` | `True` | §3.3's `POST /v4/predictions`. Safe on: it is pinned to the cheap analyses with the agent's slow passes off, so it creates no job row. |
>
> **Deletes here are soft** on versions, revisions, assessments and training jobs (v3
> parity) — the row is flagged, not removed. Users and groups are hard deletes.
>
> **If any cell fails, run §6 by hand.** A raise stops "Run All" where it happens, so
> teardown does not get its turn. Running it on its own is safe and idempotent.

## 0. Setup

In [ ]:
import base64
import json
import os
import sys
import time
from datetime import datetime, timezone

import requests
from dotenv import load_dotenv

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

# The repo's own password helpers. `verify_password` earns its place in §2.3: v3's
# /users/me leaks `hashed_password`, which lets us prove a user created through v4
# landed in the same credential store v3 hashes against.
from security_routes.utilities import hash_password, verify_password  # noqa: E402
from pathlib import Path  # noqa: E402

### 0.1 Base URL

Same convention as `integration.ipynb`: uncomment the runner you are pointing at and
comment the others. **Dev is the default** — a pre-promotion check should never need
prod credentials, and prod is by definition still running the *old* code.

In [ ]:
# Main runner (prod / `release`) — serves the PREVIOUS release, so §1's operation
# count will be the old one. Useful for a deliberate before/after, not for a pass.
# base_url = "https://tmv9bz5v4q.us-east-1.awsapprunner.com/"
# Development runner (staging / `main`) — what a promote/* branch is promoting
base_url = "https://cp3by92k8p.us-east-1.awsapprunner.com/"
# Local
# base_url = "http://localhost:8000/"

prefix = "latest"  # v3 is served under both /v3 and /latest; the UI uses /latest
V3 = f"{base_url}{prefix}"
V4 = f"{base_url}v4"

print(f"v3 -> {V3}")
print(f"v4 -> {V4}")

In [ ]:
load_dotenv("../.env", override=True)

In [ ]:
adminpassword = os.getenv("ADMIN_PASSWORD")
adminpassword is not None

In [ ]:
# Same helper integration.ipynb exposes — for minting a bcrypt hash when seeding a
# user straight into the database. Not needed for any API call below.
hash_password("password")

### 0.5 Run mode

See the warning at the top of the notebook for what each switch costs.

**One thing to know before re-running quickly.** Three separate 5/minute per-IP budgets
are in play, and only one of them is shared across the two surfaces:

- **Failed logins on `/token` are shared between v3 and v4** — the limiter keys on the
  client address, not the API version, so alternating surfaces does not double the
  allowance (#713). Successful logins are never charged (#959). This notebook makes two
  deliberate failures.
- **Account creation** and **password writes** are *not* shared: v4 uses its own named
  scopes, v3's equivalents use plain per-route limits, and the two draw on different
  counters. A full pass creates three users and makes four password writes, all on the
  v4 side, so it fits comfortably.

Back-to-back passes inside one minute may still collect a 429. That is the limiter
working, not a regression.

In [ ]:
# --- switches --------------------------------------------------------------
# OFF by default: dev and prod share one database, so a Run All should not write.
RUN_WRITE_TESTS = False

# Additionally gates §3.4. Requires RUN_WRITE_TESTS too. Real GPU work, and
# DELETE /v4/training-jobs/{id} is a 409 until the job is terminal — so an unlucky
# run leaves a row that §6 cannot clean up and will name for you.
RUN_TRAINING = False

# §3.3's fan-out. Pinned to the cheap analyses with include_translation /
# include_critique off, so `job` comes back null and no prediction job row is
# created (there is no DELETE for one). Safe to leave on.
RUN_PREDICT_FANOUT = True

# --- fixture naming --------------------------------------------------------
# Everything this notebook creates carries this prefix, so a leaked row is obvious
# and greppable in prod's UI.
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
PREFIX = f"zz-promo-{STAMP}"

# Language codes: eng / swh are the pair this repo's fixtures seed in iso_language.
# Other ISO codes fail the FK.
ISO_LANGUAGE = "eng"
ISO_LANGUAGE_ALT = "swh"
ISO_SCRIPT = "Latn"

# 41,899 lines (one per vref), 3 of them non-blank — the repo's standard tiny upload.
UPLOAD_FIXTURE = Path("../fixtures/uploadtest.txt")

print(f"fixture prefix   : {PREFIX}")
print(f"writes           : {'ENABLED' if RUN_WRITE_TESTS else 'disabled'}")
print(f"training submit  : {'ENABLED' if (RUN_WRITE_TESTS and RUN_TRAINING) else 'disabled'}")
print(f"predict fan-out  : {'ENABLED' if RUN_PREDICT_FANOUT else 'disabled'}")

### 0.6 Helpers

`record(...)` collects a pass/fail line for the summary in §7 and never raises — that is
what the exploratory cells use, so one dead endpoint does not end the run.

`compare(...)` is the opposite and is used only for the §2 parity checks: it compares
**every** mapped field, records each one, prints the table, and *then* raises if any
differed. Loud, but complete — you see all the mismatches, not just the first.

In [ ]:
# Every v4 list response is this exact envelope. Five keys, not four:
# `next_updated_since` is part of V4Page (the delta-sync cursor, §9 of the migration
# guide) and is serialized on every page, null where the read is not delta-capable.
# Pinning the whole set rather than a subset is what would catch a sixth appearing.
V4_PAGE_KEYS = {"items", "total", "limit", "offset", "next_updated_since"}

RESULTS = []  # (section, label, ok, detail)


def record(section, label, ok, detail=""):
    """Log a check and keep going. Never raises."""
    RESULTS.append((section, label, bool(ok), detail))
    print(f"{'PASS' if ok else 'FAIL'}  [{section}] {label}" + (f"  — {detail}" if detail else ""))
    return bool(ok)


def check(resp, expect=None):
    """`raise_for_status()` that shows the body, so a 422 names the field it hated."""
    if expect is not None:
        if resp.status_code != expect:
            raise RuntimeError(
                f"{resp.request.method} {resp.url} -> expected {expect}, "
                f"got {resp.status_code}: {resp.text[:500]}"
            )
        return resp
    if not resp.ok:
        raise RuntimeError(
            f"{resp.request.method} {resp.url} -> {resp.status_code}: {resp.text[:500]}"
        )
    return resp


def err(resp):
    """The v4 error envelope: {'error': {code, message, details?}} -> the inner dict."""
    try:
        return resp.json().get("error", {}) or {}
    except Exception:
        return {}


def page_all(url, headers, params=None, cap=5000):
    """Walk every page of a V4Page list and return the items.

    `total` on a v4 page counts all matching rows, ignoring limit/offset, so the walk
    knows when to stop without probing for an empty page. `cap` is a seatbelt against
    a list that grows while we are reading it.
    """
    items, offset = [], 0
    while len(items) < cap:
        page = check(requests.get(url, headers=headers, timeout=60,
                                  params={**(params or {}), "limit": 100, "offset": offset}))
        body = page.json()
        items.extend(body["items"])
        offset += body["limit"]
        if offset >= body["total"] or not body["items"]:
            return items
    return items


def _norm(value):
    """Normalise a value before comparing it across the two surfaces.

    Only timestamps need it: v3 and v4 read the same column through the same
    serializer, but a date/datetime rendered by two apps can differ in trailing
    precision or offset spelling. Parsing removes that as a source of false failure
    while keeping a real difference visible.
    """
    if isinstance(value, str):
        try:
            return datetime.fromisoformat(value.replace("Z", "+00:00"))
        except ValueError:
            return value
    return value


def diff_fields(v3_body, v4_body, field_map):
    """Mismatches between two bodies under `field_map`, as (v3_field, v4_field) pairs.

    Records nothing and raises nothing. Split out of `compare` so a caller reading a
    *live* row can peek, decide the two reads simply straddled a state change, and
    re-read before committing to a verdict.
    """
    return [
        (v3_field, v4_field)
        for v3_field, v4_field in field_map.items()
        if _norm(v3_body.get(v3_field)) != _norm(v4_body.get(v4_field))
    ]


#: v3 fields that disagree with v4 for a reason that is a **v3 defect**, not a v4
#: regression — so the notebook reports them every run and does not fail on them.
#: v3 is frozen, so these will not change; the value is knowing they are expected.
KNOWN_V3_DIVERGENCES = {
    "machineTranslation": (
        "v3's VersionOut_v3 / RevisionOut_v3 declare the field as `machineTranslation` "
        "while the column is `machine_translation`, with a `False` default — so v3 "
        "READS it as false no matter what is stored. v3's write path maps it correctly, "
        "so the stored value is right and only v3's response is wrong. Nothing live is "
        "affected today: 0 of 3,267 visible versions have the flag set. v4 reports the "
        "real value. schemas/bible.py:91 and :163."
    ),
}


def compare(section, label, v3_body, v4_body, field_map, v3_only=(), v4_only=()):
    """Assert the same row reads identically through v3 and v4.

    `field_map` is {v3 wire name: v4 wire name}. Names differ on purpose in places —
    the point is that the *values* agree, not the spellings.

    Records every field, prints a table, then raises with the full list if any
    differed. `v3_only` / `v4_only` are fields with no counterpart; they are printed
    for the record and not compared.
    """
    diffs = diff_fields(v3_body, v4_body, field_map)
    known_hits = [d for d in diffs if d[0] in KNOWN_V3_DIVERGENCES]
    mismatches = [f"{a} -> {b}: v3={v3_body.get(a)!r} v4={v4_body.get(b)!r}"
                  for a, b in diffs if a not in KNOWN_V3_DIVERGENCES]

    print(f"\n{label}")
    print(f"  {'v3 field':<24} {'v4 field':<24} {'v3 value':<28} v4 value")
    print(f"  {'-' * 24} {'-' * 24} {'-' * 28} {'-' * 28}")
    for v3_field, v4_field in field_map.items():
        a, b = v3_body.get(v3_field), v4_body.get(v4_field)
        if _norm(a) == _norm(b):
            flag = " "
        else:
            flag = "!" if v3_field in KNOWN_V3_DIVERGENCES else "✗"
        print(f"{flag} {v3_field:<24} {v4_field:<24} {str(a)[:27]:<28} {str(b)[:27]}")
    for v3_field, _ in known_hits:
        print(f"\n  ! {v3_field}: known v3 defect, not a v4 regression —")
        print(f"    {KNOWN_V3_DIVERGENCES[v3_field]}")
    for field in v3_only:
        print(f"  {field:<24} {'(v3 only)':<24} {str(v3_body.get(field))[:27]}")
    for field in v4_only:
        print(f"  {'(v4 only)':<24} {field:<24} {'':<28} {str(v4_body.get(field))[:27]}")

    record(section, label, not mismatches,
           "" if not mismatches else f"{len(mismatches)} field(s) differ")
    if mismatches:
        raise AssertionError(f"{label}: " + "; ".join(mismatches))


# --- teardown registry -----------------------------------------------------
# Every create appends here; §6 walks it in dependency order. Keeping one registry
# (rather than a finally per cell) is what makes teardown work after a cell raised
# halfway through — the ids created before the failure are still recorded.
CREATED = {
    "assessments": [],       # v4/v3 assessment ids   -> soft delete
    "revisions": [],         # revision ids           -> soft delete
    "version_grants": [],    # (version_id, group_id) -> revoke before deleting group
    "versions": [],          # version ids            -> soft delete
    "memberships": [],       # (group_id, user_id)    -> remove before deleting group
    "groups": [],            # group ids              -> hard delete
    "users": [],             # user ids               -> hard delete
    "training_jobs": [],     # training job ids       -> soft delete, terminal only
}


def track(kind, value):
    if value not in CREATED[kind]:
        CREATED[kind].append(value)
    return value


def untrack(kind, value):
    """Drop an id the cell already cleaned up, so §6 does not try again."""
    if value in CREATED[kind]:
        CREATED[kind].remove(value)
    return value


print("helpers ready")

### 0.7 Authentication on both surfaces

Worth confirming rather than assuming, since it decides how a client migrates:
`POST /v4/token` is the **same OAuth2 password grant** as v3's — form-encoded, public
(it is the endpoint that issues tokens, so it cannot require one), and the token itself
is built by the shared `auth_routes.create_access_token` with the same `{"sub",
"is_admin"}` claims. v4 keeps vetted client credentials; there is no OAuth redirect flow.

That makes the tokens **byte-compatible**, which is worth pinning rather than assuming:
a token minted at `/v4/token` must work on v3 routes and vice versa. If that ever stops
being true, every client mid-migration breaks.

In [ ]:
# v3 admin token — exactly integration.ipynb's call.
r = check(requests.post(f"{V3}/token", timeout=30,
                       data={"username": "admin", "password": adminpassword}), expect=200)
v3_token = r.json()["access_token"]
V3_ADMIN = {"Authorization": f"Bearer {v3_token}"}

# v4 admin token — same grant, same form encoding, different path.
r = check(requests.post(f"{V4}/token", timeout=30,
                       data={"username": "admin", "password": adminpassword}), expect=200)
v4_tok = r.json()
v4_token = v4_tok["access_token"]
V4_ADMIN = {"Authorization": f"Bearer {v4_token}"}

record("0", "POST /v4/token issues a bearer token", v4_tok.get("token_type") == "bearer",
       f"{len(v4_token)} chars")
record("0", "v4 token response is exactly {access_token, token_type}",
       set(v4_tok) == {"access_token", "token_type"}, str(sorted(v4_tok)))

In [ ]:
# Cross-surface: each token must be accepted by the other surface.
cross_v4 = requests.get(f"{V4}/users/me", headers=V3_ADMIN, timeout=30)
cross_v3 = requests.get(f"{V3}/users/me", headers=V4_ADMIN, timeout=30)

record("0", "a v3 token authenticates a v4 route", cross_v4.status_code == 200,
       str(cross_v4.status_code))
record("0", "a v4 token authenticates a v3 route", cross_v3.status_code == 200,
       str(cross_v3.status_code))

me = check(requests.get(f"{V4}/users/me", headers=V4_ADMIN, timeout=30), expect=200).json()
ADMIN_USER_ID = me["id"]
record("0", "authenticated account is an admin", bool(me.get("is_admin")),
       f"user {ADMIN_USER_ID} ({me['username']})")
print(json.dumps(me, indent=2))

## 1. v4 surface discovery

`GET /v4/` is the one deliberately public v4 route — if it needs a token, the mount is
misconfigured.

The operation count is the promotion's own headline claim. **PR #964** ("Promote main →
release: the rest of the v4 surface") states twenty-three new endpoints taking v4 from
**33 to 56**, both counts including the discovery page, and §6 of that PR lists
`GET /v4/openapi.json` reporting 56 as the thing to watch after deploying. So that is the
number asserted below, and if you are promoting something else, re-read the PR body and
change it — do not just make the test pass.

Pointed at **prod** this cell is *expected* to report 33 until #964 merges.

In [ ]:
r = requests.get(f"{V4}/", timeout=30)
print(r.status_code, r.text[:300])
record("1", "GET /v4/ answers without a token", r.status_code == 200)
record("1", "discovery payload is {version, status}",
       r.ok and set(r.json()) == {"version", "status"},
       r.text[:120])

In [ ]:
# Expected operation count. Source: PR #964 §1 ("taking the total from 33 to 56") and
# §6.4 ("GET /v4/openapi.json should report 56 operations"). Update this alongside the
# promotion PR it tracks, never to silence a failure.
EXPECTED_V4_OPERATIONS = 56
PROMOTION_PR = "#964"

METHODS = {"get", "post", "put", "patch", "delete"}

spec = check(requests.get(f"{V4}/openapi.json", timeout=60), expect=200).json()
v4_ops = sorted(
    (method.upper(), path)
    for path, item in spec["paths"].items()
    for method in item
    if method in METHODS
)

print(f"{spec['info']['title']} — {spec['info']['version']}")
print(f"{len(v4_ops)} operations\n")
for method, path in v4_ops:
    print(f"  {method:<7} /v4{path}")

record("1", f"v4 publishes {EXPECTED_V4_OPERATIONS} operations (PR {PROMOTION_PR})",
       len(v4_ops) == EXPECTED_V4_OPERATIONS,
       f"got {len(v4_ops)}")

In [ ]:
# v3 must be untouched by the promotion. PR #964 §3: "256 operations before, 256 after."
EXPECTED_V3_OPERATIONS = 256

v3_spec = check(requests.get(f"{base_url}openapi.json", timeout=60), expect=200).json()
v3_ops = [
    (method.upper(), path)
    for path, item in v3_spec["paths"].items()
    for method in item
    if method in METHODS
]
record("1", f"v3 still publishes {EXPECTED_V3_OPERATIONS} operations",
       len(v3_ops) == EXPECTED_V3_OPERATIONS, f"got {len(v3_ops)}")

# v4 is mounted as its own sub-app, so it must not appear in v3's schema at all.
leaked = [p for p in v3_spec["paths"] if p.startswith("/v4/")]
record("1", "v4 paths do not leak into the main app's schema", not leaked, str(leaked[:5]))

## 2. v3 / v4 parity

The core of this notebook. For every resource that exists on both surfaces: write it
through one, read it back through the other, and assert the values agree field for field.

**The spellings differ on purpose in places, and that is not what is being tested.** Two
distinct sets of renames are in play, and they are easy to conflate:

*Longstanding v3 → v4 differences* (v4 is snake_case throughout, v3 is not):

| v3 | v4 |
|---|---|
| `machineTranslation` | `machine_translation` |
| `add_to_groups` (request) | `group_ids` (request) |
| `bible_version_id` (revision) | `version_id` |
| `date` (revision) | `uploaded_date` |
| `kwargs` (assessment) | `options` |
| `status` (assessment) | `state` — *and a different vocabulary*, see §2.5 |

*The #925 renames this promotion carries* — these changed **within v4**, old v4 → new v4,
so they are not a v3/v4 difference at all. v3 already emitted `forward_translation_id` and
`back_translation_id` in its responses; what moved was v4's spelling of the same fields,
plus the four assessment timestamps:

| old v4 | new v4 |
|---|---|
| `forward_translation` | `forward_translation_id` |
| `back_translation` | `back_translation_id` |
| `date` | `created_at` |
| `requested_time` | `requested_at` |
| `start_time` | `started_at` |
| `end_time` | `ended_at` |

The maps below encode all of it. What is asserted is that the **same underlying row**
reads the same through both surfaces.

In [ ]:
# v3 wire name -> v4 wire name. Exhaustive over v3's VersionOut_v3.
VERSION_FIELD_MAP = {
    "id": "id",
    "name": "name",
    "iso_language": "iso_language",
    "iso_script": "iso_script",
    "abbreviation": "abbreviation",
    "rights": "rights",
    # Already `_id` on v3's *response*; #925 renamed v4's half to match.
    "forward_translation_id": "forward_translation_id",
    "back_translation_id": "back_translation_id",
    "machineTranslation": "machine_translation",
    "owner_id": "owner_id",
    "group_ids": "group_ids",
    "is_reference": "is_reference",
    "transcribed_audio": "transcribed_audio",
    "deleted": "deleted",
    "updated_at": "updated_at",
}

REVISION_FIELD_MAP = {
    "id": "id",
    "bible_version_id": "version_id",
    "version_abbreviation": "version_abbreviation",
    "date": "uploaded_date",
    "name": "name",
    "published": "published",
    "back_translation_id": "back_translation_id",
    "machineTranslation": "machine_translation",
    "iso_language": "iso_language",
    "deleted": "deleted",
    "updated_at": "updated_at",
}

ASSESSMENT_FIELD_MAP = {
    "id": "id",
    "revision_id": "revision_id",
    "reference_id": "reference_id",
    "type": "type",
    "requested_time": "requested_at",
    "start_time": "started_at",
    "end_time": "ended_at",
    "owner_id": "owner_id",
    "status_detail": "status_detail",
    "percent_complete": "percent_complete",
    "kwargs": "options",
    "deleted": "deleted",
    "updated_at": "updated_at",
}

# v3's free-text `status` -> v4's closed JobState vocabulary (api_v4/jobs.py).
STATUS_TO_STATE = {
    "queued": "PENDING",
    "running": "RUNNING",
    "finished": "SUCCEEDED",
    "failed": "FAILED",
}

# `note` is deliberately absent. v3's /result builds every row through a GROUP BY that
# selects only min(id), assessment_id, the group columns, avg(score), bool_or(flag) and
# bool_or(hide) — so v3 reports `note` as null for every assessment of every type. A
# comparison could therefore never catch a regression, and would raise a false parity
# failure the moment v4 correctly returned a real note. It is listed as v3-only instead.
RESULT_FIELD_MAP = {
    "id": "id",
    "assessment_id": "assessment_id",
    "vref": "vref",
    "score": "score",
    "flag": "flag",
    "hide": "hide",
}

print("field maps loaded")

### 2.0 Fixture group

Both surfaces refuse to create a version unless the caller is a member of every group it
is being added to, and **being an admin is not an exemption** on either. So the parity
work starts by making a throwaway group and putting the admin in it.

This also exercises two of the endpoints this promotion adds: `POST /v4/groups` and
`PUT /v4/groups/{id}/members/{user_id}`.

In [ ]:
GROUP_ID = None

if not RUN_WRITE_TESTS:
    print("skipped — RUN_WRITE_TESTS is False (§0.5)")
else:
    r = check(requests.post(
        f"{V4}/groups",
        json={"name": f"{PREFIX}-grp", "description": "pre-promotion parity fixture"},
        headers=V4_ADMIN, timeout=30,
    ), expect=201)
    group = r.json()
    GROUP_ID = track("groups", group["id"])
    record("2.0", "POST /v4/groups -> 201", True, f"group {GROUP_ID}")
    record("2.0", "group body is {id, name, description}",
           set(group) == {"id", "name", "description"}, str(sorted(group)))

    r = requests.put(f"{V4}/groups/{GROUP_ID}/members/{ADMIN_USER_ID}",
                     headers=V4_ADMIN, timeout=30)
    record("2.0", "PUT group member -> 204", r.status_code == 204, str(r.status_code))
    if r.status_code == 204:
        track("memberships", (GROUP_ID, ADMIN_USER_ID))

    # Idempotent by contract: adding an existing member is another 204, not a 400.
    again = requests.put(f"{V4}/groups/{GROUP_ID}/members/{ADMIN_USER_ID}",
                         headers=V4_ADMIN, timeout=30)
    record("2.0", "re-adding an existing member is idempotent (204)",
           again.status_code == 204, str(again.status_code))

    # And the membership is visible on the read side.
    mine = page_all(f"{V4}/users/me/groups", V4_ADMIN)
    record("2.0", "new group appears in GET /v4/users/me/groups",
           GROUP_ID in [g["id"] for g in mine], f"{len(mine)} group(s)")

### 2.1 Versions

Created through v4, read back through v3, and the reverse. Both directions matter: the
v4-created row proves v3 can still see what v4 writes (the website reads v3), and the
v3-created row proves v4 reports what the website writes.

In [ ]:
V4_VERSION_ID = None
V3_VERSION_ID = None

if not RUN_WRITE_TESTS or GROUP_ID is None:
    print("skipped — needs RUN_WRITE_TESTS and the §2.0 group")
else:
    # ---- created through v4 -------------------------------------------------
    body = {
        "name": f"{PREFIX}-v4-version",
        "iso_language": ISO_LANGUAGE,
        "iso_script": ISO_SCRIPT,
        "abbreviation": f"{PREFIX[:12]}4",
        "rights": "pre-promotion fixture",
        "machine_translation": True,
        "is_reference": False,
        "transcribed_audio": False,
        "group_ids": [GROUP_ID],
    }
    v4_version = check(requests.post(f"{V4}/versions", json=body, headers=V4_ADMIN,
                                     timeout=60), expect=201).json()
    V4_VERSION_ID = track("versions", v4_version["id"])
    track("version_grants", (V4_VERSION_ID, GROUP_ID))
    record("2.1", "POST /v4/versions -> 201", True, f"version {V4_VERSION_ID}")

    # Read the same row back through v3's list.
    v3_list = check(requests.get(f"{V3}/version", headers=V3_ADMIN, timeout=60)).json()
    v3_view = next((v for v in v3_list if v["id"] == V4_VERSION_ID), None)
    assert v3_view is not None, f"version {V4_VERSION_ID} created on v4 is invisible to v3"

    compare("2.1", "version created on v4 reads identically on v3",
            v3_view, v4_version, VERSION_FIELD_MAP)

In [ ]:
if not RUN_WRITE_TESTS or GROUP_ID is None:
    print("skipped — needs RUN_WRITE_TESTS and the §2.0 group")
else:
    # ---- created through v3 -------------------------------------------------
    # v3's request spelling: forwardTranslation / backTranslation / machineTranslation
    # / add_to_groups. The response comes back with the *_id spellings regardless.
    #
    # Note for anyone editing this: v3's create commits the version row *before* it
    # validates add_to_groups, so a rejected group check leaves an orphan row behind
    # with no id returned to clean it up. This call cannot trip that — GROUP_ID is a
    # group §2.0 just created and put this caller in — but do not point it at a group
    # the caller may not belong to.
    body = {
        "name": f"{PREFIX}-v3-version",
        "iso_language": ISO_LANGUAGE_ALT,
        "iso_script": ISO_SCRIPT,
        "abbreviation": f"{PREFIX[:12]}3",
        "rights": "pre-promotion fixture",
        "machineTranslation": False,
        "is_reference": True,
        "transcribed_audio": False,
        "add_to_groups": [GROUP_ID],
    }
    v3_version = check(requests.post(f"{V3}/version", json=body, headers=V3_ADMIN,
                                     timeout=60)).json()
    V3_VERSION_ID = track("versions", v3_version["id"])
    track("version_grants", (V3_VERSION_ID, GROUP_ID))
    record("2.1", "POST /latest/version (v3) succeeded", True, f"version {V3_VERSION_ID}")

    v4_view = check(requests.get(f"{V4}/versions/{V3_VERSION_ID}", headers=V4_ADMIN,
                                 timeout=30), expect=200).json()

    # Re-read through v3's GET rather than comparing v4 against v3's POST body.
    # v3's create commits and refreshes the row *before* it writes the group-access
    # rows, so its POST response always reports `group_ids: []` however many groups
    # were granted. The grants themselves are real — v3's own GET shows them. Comparing
    # the POST body would report a stale response as a data divergence.
    v3_reread = next(
        v for v in check(requests.get(f"{V3}/version", headers=V3_ADMIN, timeout=60)).json()
        if v["id"] == V3_VERSION_ID
    )
    record("2.1", "v3's POST body under-reports group_ids (its GET is correct)",
           v3_version.get("group_ids") == [] and v3_reread.get("group_ids") == [GROUP_ID],
           f"POST said {v3_version.get('group_ids')}, GET says {v3_reread.get('group_ids')}")

    compare("2.1", "version created on v3 reads identically on v4",
            v3_reread, v4_view, VERSION_FIELD_MAP)

    # The v4 body must be closed snake_case: no v3 spelling survives into it.
    record("2.1", "v4 response carries no camelCase survivors",
           "machineTranslation" not in v4_view and "machine_translation" in v4_view)
    record("2.1", "v4 response uses the #925 *_id spellings",
           "forward_translation_id" in v4_view and "forward_translation" not in v4_view,
           str(sorted(k for k in v4_view if "translation" in k)))

### 2.2 Revisions

The two surfaces take the verse text in **completely different ways** — v3 as a
`multipart/form-data` file upload, v4 as base64 inside a JSON body — so this is the one
parity check where the write paths share no code above the service layer. The row they
produce still has to read the same.

`fixtures/uploadtest.txt` is the repo's standard tiny upload: 41,899 lines, one per vref,
three of them non-blank.

In [ ]:
V3_REVISION_ID = None
V4_REVISION_ID = None

if not RUN_WRITE_TESTS or V3_VERSION_ID is None or V4_VERSION_ID is None:
    print("skipped — needs RUN_WRITE_TESTS and the §2.1 versions")
else:
    # ---- created through v3: multipart upload --------------------------------
    with open(UPLOAD_FIXTURE, "rb") as fh:
        r = check(requests.post(
            f"{V3}/revision",
            params={"version_id": V3_VERSION_ID, "name": f"{PREFIX}-v3-rev"},
            files={"file": fh},
            headers=V3_ADMIN, timeout=180,
        ))
    v3_revision = r.json()
    V3_REVISION_ID = track("revisions", v3_revision["id"])
    record("2.2", "POST /latest/revision (multipart) succeeded", True,
           f"revision {V3_REVISION_ID}")

    v4_view = check(requests.get(f"{V4}/revisions/{V3_REVISION_ID}", headers=V4_ADMIN,
                                 timeout=30), expect=200).json()

    # Re-read through v3's GET rather than comparing against v3's POST body, for the
    # same reason §2.1 does. v3's create commits and then builds its response from the
    # in-memory row without refreshing it, so the `updated_at` the database trigger
    # stamped never reaches the POST body — it reports null. The stored value is
    # correct and v3's own GET returns it.
    v3_reread = next(
        x for x in check(requests.get(f"{V3}/revision",
                                      params={"version_id": V3_VERSION_ID},
                                      headers=V3_ADMIN, timeout=60)).json()
        if x["id"] == V3_REVISION_ID
    )
    record("2.2", "v3's POST body omits updated_at (its GET has it)",
           v3_revision.get("updated_at") is None
           and v3_reread.get("updated_at") is not None,
           f"POST said {v3_revision.get('updated_at')}, "
           f"GET says {v3_reread.get('updated_at')}")

    compare("2.2", "revision uploaded on v3 reads identically on v4",
            v3_reread, v4_view, REVISION_FIELD_MAP,
            v3_only=("is_reference",))

In [ ]:
if not RUN_WRITE_TESTS or V4_VERSION_ID is None:
    print("skipped — needs RUN_WRITE_TESTS and the §2.1 v4 version")
else:
    # ---- created through v4: inline base64 -----------------------------------
    content_b64 = base64.b64encode(UPLOAD_FIXTURE.read_bytes()).decode("ascii")
    body = {
        "version_id": V4_VERSION_ID,
        "name": f"{PREFIX}-v4-rev",
        "published": False,
        "machine_translation": False,
        "text": {"type": "inline", "content_base64": content_b64},
    }
    v4_revision = check(requests.post(f"{V4}/revisions", json=body, headers=V4_ADMIN,
                                      timeout=180), expect=201).json()
    V4_REVISION_ID = track("revisions", v4_revision["id"])
    record("2.2", "POST /v4/revisions (inline base64) -> 201", True,
           f"revision {V4_REVISION_ID}")

    v3_list = check(requests.get(f"{V3}/revision", params={"version_id": V4_VERSION_ID},
                                 headers=V3_ADMIN, timeout=60)).json()
    v3_view = next((x for x in v3_list if x["id"] == V4_REVISION_ID), None)
    assert v3_view is not None, f"revision {V4_REVISION_ID} created on v4 is invisible to v3"

    compare("2.2", "revision created on v4 reads identically on v3",
            v3_view, v4_revision, REVISION_FIELD_MAP, v3_only=("is_reference",))

    # The verse text itself must have landed the same way through both writers.
    v4_text = check(requests.get(f"{V4}/revisions/{V4_REVISION_ID}/text",
                                 headers=V4_ADMIN, timeout=120)).text
    # v3's equivalent is not raw text: GET /latest/text returns a JSON list of verse
    # objects, and `include_verses=union` gives exactly the verses that have text. So
    # the comparable quantity is "how many verses carry text", not the line count.
    # Not wrapped in check(): the point is to report what v3 did, which a raising
    # helper would turn into a check that can only pass.
    v4_lines = v4_text.splitlines()
    v4_with_text = [line for line in v4_lines if line.strip()]
    v3_text = requests.get(f"{V3}/text",
                           params={"revision_id": V4_REVISION_ID, "include_verses": "union"},
                           headers=V3_ADMIN, timeout=120)

    record("2.2", "GET /v4/revisions/{id}/text returns one line per vref",
           len(v4_lines) == 41899, f"{len(v4_lines)} lines")
    record("2.2", "the same revision's text is readable on v3", v3_text.ok,
           f"{v3_text.status_code} {v3_text.text[:120]}")
    if v3_text.ok:
        record("2.2", "both surfaces agree how many verses carry text",
               len(v3_text.json()) == len(v4_with_text),
               f"v3={len(v3_text.json())} v4={len(v4_with_text)}")

### 2.3 Users

Neither surface has a "read any user" endpoint — v3 has no `GET /users` list at all, and
v4's user reads are all self-scoped (`/v4/users/me`). So parity here is proved by
**logging in as the created account on the other surface**: a user created through v3 must
be able to authenticate against v4 and see itself, and vice versa.

The v4-created direction gets one extra check. v3's `/users/me` hands back the raw ORM
object, `hashed_password` included — the leak v4's four-field allowlist exists to fix
(#859). That leak is exactly what lets us prove the password v4 stored is the one we
sent, hashed by the same helper v3 verifies against: one credential store, two surfaces.

In [ ]:
V3_MADE_USER_ID = None
V4_MADE_USER_ID = None
V3_MADE_USERNAME = f"{PREFIX}-u3"
V4_MADE_USERNAME = f"{PREFIX}-u4"
FIXTURE_PASSWORD = "promo-notebook-pw-1"

if not RUN_WRITE_TESTS:
    print("skipped — RUN_WRITE_TESTS is False (§0.5)")
else:
    # ---- created through v3 (username/email as query params, password as form) ----
    r = check(requests.post(
        f"{V3}/users",
        params={"username": V3_MADE_USERNAME, "email": f"{V3_MADE_USERNAME}@example.com",
                "is_admin": False},
        data={"username": V3_MADE_USERNAME, "password": FIXTURE_PASSWORD},
        headers=V3_ADMIN, timeout=30,
    ))
    v3_user = r.json()
    V3_MADE_USER_ID = track("users", v3_user["id"])
    record("2.3", "POST /latest/users (v3) succeeded", True, f"user {V3_MADE_USER_ID}")

    # That account must authenticate against v4 and see itself.
    tok = check(requests.post(f"{V4}/token",
                              data={"username": V3_MADE_USERNAME, "password": FIXTURE_PASSWORD},
                              timeout=30), expect=200).json()
    as_v3_user = {"Authorization": f"Bearer {tok['access_token']}"}
    v4_me = check(requests.get(f"{V4}/users/me", headers=as_v3_user, timeout=30),
                  expect=200).json()

    compare("2.3", "user created on v3 reads identically on v4",
            v3_user, v4_me,
            {"id": "id", "username": "username", "email": "email", "is_admin": "is_admin"})
    record("2.3", "v4 users/me is the closed 4-field allowlist",
           set(v4_me) == {"id", "username", "email", "is_admin"}, str(sorted(v4_me)))

In [ ]:
if not RUN_WRITE_TESTS:
    print("skipped — RUN_WRITE_TESTS is False (§0.5)")
else:
    # ---- created through v4 (one JSON body) ----------------------------------
    v4_user = check(requests.post(
        f"{V4}/users",
        json={"username": V4_MADE_USERNAME, "email": f"{V4_MADE_USERNAME}@example.com",
              "password": FIXTURE_PASSWORD},
        headers=V4_ADMIN, timeout=30,
    ), expect=201).json()
    V4_MADE_USER_ID = track("users", v4_user["id"])
    record("2.3", "POST /v4/users -> 201", True, f"user {V4_MADE_USER_ID}")
    record("2.3", "v4-created account is never an admin", v4_user["is_admin"] is False)

    # The same account authenticating against v3.
    tok = check(requests.post(f"{V3}/token",
                              data={"username": V4_MADE_USERNAME, "password": FIXTURE_PASSWORD},
                              timeout=30), expect=200).json()
    as_v4_user = {"Authorization": f"Bearer {tok['access_token']}"}
    v3_me = check(requests.get(f"{V3}/users/me", headers=as_v4_user, timeout=30),
                  expect=200).json()

    compare("2.3", "user created on v4 reads identically on v3",
            v3_me, v4_user,
            {"id": "id", "username": "username", "email": "email", "is_admin": "is_admin"})

    # v3 leaks the hash; use it to prove both surfaces share one credential store.
    # The leak itself is not recorded as a check — v3 losing it would be an
    # improvement, not a regression, and a FAIL line would say the opposite.
    leaked = v3_me.get("hashed_password")
    if leaked:
        print("  v3 /users/me leaks hashed_password — the #859 bug v4's allowlist "
              "fixes, and what makes the next check possible")
        record("2.3", "password stored by v4 verifies against v3's hasher",
               verify_password(FIXTURE_PASSWORD, leaked))
    else:
        record("2.3", "password stored by v4 verifies against v3's hasher", True,
               "not checked — v3 no longer exposes hashed_password")

### 2.4 Groups

v3 lists groups as a bare array; v4 wraps every list in the `V4Page` envelope
(`items`/`total`/`limit`/`offset`, plus the `next_updated_since` delta cursor). Same rows
underneath.

In [ ]:
V3_MADE_GROUP_NAME = f"{PREFIX}-g3"
V3_MADE_GROUP_ID = None

if not RUN_WRITE_TESTS:
    print("skipped — RUN_WRITE_TESTS is False (§0.5)")
else:
    v3_group = check(requests.post(
        f"{V3}/groups",
        params={"name": V3_MADE_GROUP_NAME, "description": "pre-promotion fixture"},
        headers=V3_ADMIN, timeout=30,
    )).json()
    V3_MADE_GROUP_ID = track("groups", v3_group["id"])
    record("2.4", "POST /latest/groups (v3) succeeded", True, f"group {V3_MADE_GROUP_ID}")

    # One page for the envelope check, then the whole list to find the row.
    page = check(requests.get(f"{V4}/groups", params={"limit": 100}, headers=V4_ADMIN,
                              timeout=60)).json()
    all_groups = page_all(f"{V4}/groups", V4_ADMIN)
    found = next((g for g in all_groups if g["id"] == V3_MADE_GROUP_ID), None)
    assert found is not None, f"group {V3_MADE_GROUP_ID} created on v3 is invisible to v4"

    compare("2.4", "group created on v3 reads identically on v4", v3_group, found,
            {"id": "id", "name": "name", "description": "description"})
    record("2.4", "v4 group list uses the V4Page envelope",
           set(page) == V4_PAGE_KEYS, str(sorted(page)))

    # And the §2.0 group, created on v4, must be visible to v3.
    if GROUP_ID is not None:
        v3_groups = check(requests.get(f"{V3}/groups", headers=V3_ADMIN, timeout=60)).json()
        mirror = next((g for g in v3_groups if g["id"] == GROUP_ID), None)
        record("2.4", "group created on v4 is visible to v3", mirror is not None,
               str(mirror))

### 2.5 Assessments

The renamed half of the promotion. An assessment created through v3 is read back through
v4, where four timestamp fields and the status field answer to different names — and
`state` answers with a **different vocabulary**, not just a different spelling: v3's
free-text `queued`/`running`/`finished`/`failed` becomes v4's closed uppercase `JobState`.

`sentence-length` is the cheapest type to submit and the fixture revision is three verses
of filler, so this costs essentially nothing. It does dispatch a real Modal run, which is
what makes it a live check rather than a schema comparison.

In [ ]:
V3_ASSESSMENT_ID = None

if not RUN_WRITE_TESTS or V3_REVISION_ID is None:
    print("skipped — needs RUN_WRITE_TESTS and the §2.2 revision")
else:
    r = check(requests.post(
        f"{V3}/assessment",
        params={"revision_id": V3_REVISION_ID, "reference_id": V3_REVISION_ID,
                "type": "sentence-length"},
        headers=V3_ADMIN, timeout=120,
    ))
    v3_assessment = r.json()[0]
    V3_ASSESSMENT_ID = track("assessments", v3_assessment["id"])
    record("2.5", "POST /latest/assessment (v3) succeeded", True,
           f"assessment {V3_ASSESSMENT_ID} status={v3_assessment.get('status')}")

    # The run is live: `status`, the timestamps and `updated_at` all move while it
    # goes queued -> running -> finished. Two reads that straddle a transition would
    # disagree for a reason that is not a parity bug, so read both back to back and
    # re-read if they differ. Three attempts, then the difference is real.
    for attempt in range(3):
        # v4 answers 202 while the job is still PENDING, 200 once it is not. Both
        # carry the same body, so accept either rather than pinning one.
        r = requests.get(f"{V4}/assessments/{V3_ASSESSMENT_ID}", headers=V4_ADMIN,
                         timeout=30)
        v4_assessment = check(r).json()
        v3_now = check(requests.get(f"{V3}/assessment", params={"id": V3_ASSESSMENT_ID},
                                    headers=V3_ADMIN, timeout=30)).json()[0]
        if not diff_fields(v3_now, v4_assessment, ASSESSMENT_FIELD_MAP) or attempt == 2:
            break
        print("  the run moved between the two reads — re-reading")
        time.sleep(5)

    record("2.5", "GET /v4/assessments/{id} answers 200 or 202",
           r.status_code in (200, 202), str(r.status_code))

    compare("2.5", "assessment created on v3 reads identically on v4",
            v3_now, v4_assessment, ASSESSMENT_FIELD_MAP,
            v3_only=("is_training", "attempt_count"),
            v4_only=("job_id", "state", "result", "error"))

    # status -> state is a vocabulary change, so it is checked separately.
    expected_state = STATUS_TO_STATE.get(v3_now.get("status"))
    record("2.5", "v3 status maps onto v4 state",
           v4_assessment.get("state") == expected_state,
           f"v3 status={v3_now.get('status')!r} -> v4 state={v4_assessment.get('state')!r}")

    # The poll body is the resource plus the four job-envelope keys.
    record("2.5", "poll body carries the job envelope",
           {"job_id", "state", "result", "error"} <= set(v4_assessment),
           str(sorted(set(v4_assessment) & {"job_id", "state", "result", "error"})))
    record("2.5", "job_id is the assessment id as a string",
           v4_assessment.get("job_id") == str(V3_ASSESSMENT_ID))
    record("2.5", "v4 drops v3's three internal fields",
           not ({"status", "is_training", "attempt_count"} & set(v4_assessment)))

### 2.6 Results

v3 returns `{"results": [...], "total_count": N}`; v4 returns the `V4Page` envelope —
`items`/`total`/`limit`/`offset` plus the `next_updated_since` delta cursor. The rows
inside must match for the fields both carry.

Result rows only exist once the run finishes, so this cell tolerates an empty read — an
empty result set on a just-submitted assessment is the run still going, not a regression.
Re-run the cell in a minute if you want the populated comparison.

In [ ]:
if not RUN_WRITE_TESTS or V3_ASSESSMENT_ID is None:
    print("skipped — needs RUN_WRITE_TESTS and the §2.5 assessment")
else:
    v3_results = check(requests.get(f"{V3}/result",
                                    params={"assessment_id": V3_ASSESSMENT_ID},
                                    headers=V3_ADMIN, timeout=120)).json()
    v4_results = check(requests.get(f"{V4}/assessments/{V3_ASSESSMENT_ID}/results",
                                    headers=V4_ADMIN, timeout=120)).json()

    record("2.6", "v3 result body is {results, total_count}",
           set(v3_results) == {"results", "total_count"}, str(sorted(v3_results)))
    record("2.6", "v4 result body is the V4Page envelope",
           set(v4_results) == V4_PAGE_KEYS, str(sorted(v4_results)))
    record("2.6", "v4 result page defaults to limit=100 (ResultPaginationParams)",
           v4_results["limit"] == 100, f"limit={v4_results['limit']}")

    print(f"v3 total_count={v3_results['total_count']}  v4 total={v4_results['total']}")

    if not v3_results["results"] and not v4_results["items"]:
        record("2.6", "results comparison", True,
               "both empty — the run has not finished; re-run this cell later")
    else:
        record("2.6", "both surfaces report the same row count",
               v3_results["total_count"] == v4_results["total"],
               f"v3={v3_results['total_count']} v4={v4_results['total']}")

        by_vref_v3 = {row["vref"]: row for row in v3_results["results"]}
        by_vref_v4 = {row["vref"]: row for row in v4_results["items"]}
        shared = sorted(set(by_vref_v3) & set(by_vref_v4))

        # v3 returns every row it has; v4 returns one page. Only when the whole result
        # set fits in that page are the two sets comparable — otherwise all that can be
        # asserted is that v4's page is drawn from the same rows.
        if v4_results["total"] <= v4_results["limit"]:
            record("2.6", "the two surfaces agree on which vrefs have results",
                   set(by_vref_v3) == set(by_vref_v4),
                   f"{len(shared)} shared, v3-only={len(set(by_vref_v3) - set(by_vref_v4))}, "
                   f"v4-only={len(set(by_vref_v4) - set(by_vref_v3))}")
        else:
            record("2.6", "v4's first page of results is a subset of v3's rows",
                   set(by_vref_v4) <= set(by_vref_v3),
                   f"{v4_results['total']} rows, comparing the first "
                   f"{v4_results['limit']}")
        if shared:
            probe = shared[0]
            compare("2.6", f"result row {probe} reads identically on both surfaces",
                    by_vref_v3[probe], by_vref_v4[probe], RESULT_FIELD_MAP,
                    v3_only=("source", "target", "revision_text", "reference_text",
                             "note"),
                    v4_only=("vrefs", "note"))

## 3. v4-only lifecycles

Each vertical slice walked end to end with real calls, not just reads.

### 3.1 Users and groups — the full write lifecycle

Eight of the twenty-three endpoints this promotion adds live here. Until now v4 could only
*read* users and groups; creating one meant dropping back to v3.

Everything created in this cell is also deleted in it, so it leaves nothing for §6 —
which is the point: `DELETE /v4/users/{id}` refuses with a 409 while anything still names
the user as an owner, and **a soft-deleted version still counts**. So the throwaway user
must never own a version, and this cell is careful not to give it one.

In [ ]:
if not RUN_WRITE_TESTS:
    print("skipped — RUN_WRITE_TESTS is False (§0.5)")
else:
    lc_user_name = f"{PREFIX}-lc-user"
    lc_pw_1, lc_pw_2, lc_pw_3 = "lifecycle-pw-1", "lifecycle-pw-2", "lifecycle-pw-3"
    lc_user_id = lc_group_id = None

    try:
        # ---- create ---------------------------------------------------------
        user = check(requests.post(f"{V4}/users", headers=V4_ADMIN, timeout=30, json={
            "username": lc_user_name, "email": f"{lc_user_name}@example.com",
            "password": lc_pw_1,
        }), expect=201).json()
        lc_user_id = track("users", user["id"])
        record("3.1", "POST /v4/users -> 201", True, f"user {lc_user_id}")

        group = check(requests.post(f"{V4}/groups", headers=V4_ADMIN, timeout=30, json={
            "name": f"{PREFIX}-lc-group", "description": "lifecycle fixture",
        }), expect=201).json()
        lc_group_id = track("groups", group["id"])
        record("3.1", "POST /v4/groups -> 201", True, f"group {lc_group_id}")

        # A duplicate name is a 409 with a machine-readable code, not a 500.
        dup = requests.post(f"{V4}/groups", headers=V4_ADMIN, timeout=30,
                            json={"name": f"{PREFIX}-lc-group"})
        record("3.1", "duplicate group name -> 409 GROUP_NAME_TAKEN",
               dup.status_code == 409 and err(dup).get("code") == "GROUP_NAME_TAKEN",
               f"{dup.status_code} {err(dup).get('code')}")

        # ---- membership ------------------------------------------------------
        r = requests.put(f"{V4}/groups/{lc_group_id}/members/{lc_user_id}",
                         headers=V4_ADMIN, timeout=30)
        record("3.1", "PUT /v4/groups/{id}/members/{user_id} -> 204",
               r.status_code == 204, str(r.status_code))
        if r.status_code == 204:
            track("memberships", (lc_group_id, lc_user_id))

        # Seen from the member's own token.
        tok = check(requests.post(f"{V4}/token",
                                  data={"username": lc_user_name, "password": lc_pw_1},
                                  timeout=30), expect=200).json()
        as_member = {"Authorization": f"Bearer {tok['access_token']}"}
        mine = check(requests.get(f"{V4}/users/me/groups", headers=as_member,
                                  timeout=30)).json()
        record("3.1", "the new member sees the group on /v4/users/me/groups",
               [g["id"] for g in mine["items"]] == [lc_group_id], str(mine["items"]))

        # A non-admin must not be able to list every group.
        forbidden = requests.get(f"{V4}/groups", headers=as_member, timeout=30)
        record("3.1", "non-admin GET /v4/groups -> 403 ADMIN_REQUIRED",
               forbidden.status_code == 403
               and err(forbidden).get("code") == "ADMIN_REQUIRED",
               f"{forbidden.status_code} {err(forbidden).get('code')}")

        # ---- password: self-service ------------------------------------------
        r = requests.post(f"{V4}/users/me/password", headers=as_member, timeout=30,
                          json={"current_password": lc_pw_1, "new_password": lc_pw_2})
        record("3.1", "POST /v4/users/me/password -> 204", r.status_code == 204,
               str(r.status_code))

        wrong = requests.post(f"{V4}/users/me/password", headers=as_member, timeout=30,
                              json={"current_password": "not-it", "new_password": lc_pw_3})
        record("3.1", "wrong current password -> 403 INCORRECT_PASSWORD",
               wrong.status_code == 403 and err(wrong).get("code") == "INCORRECT_PASSWORD",
               f"{wrong.status_code} {err(wrong).get('code')}")

        old = requests.post(f"{V4}/token",
                            data={"username": lc_user_name, "password": lc_pw_1}, timeout=30)
        new = requests.post(f"{V4}/token",
                            data={"username": lc_user_name, "password": lc_pw_2}, timeout=30)
        record("3.1", "the old password stops working", old.status_code == 401,
               str(old.status_code))
        record("3.1", "the new password works", new.status_code == 200, str(new.status_code))

        # ---- password: admin reset -------------------------------------------
        r = requests.put(f"{V4}/users/{lc_user_id}/password", headers=V4_ADMIN, timeout=30,
                         json={"new_password": lc_pw_3})
        record("3.1", "PUT /v4/users/{id}/password (admin reset) -> 204",
               r.status_code == 204, str(r.status_code))
        after = requests.post(f"{V4}/token",
                              data={"username": lc_user_name, "password": lc_pw_3}, timeout=30)
        record("3.1", "the reset password works", after.status_code == 200,
               str(after.status_code))

        # A non-admin must not be able to reset anyone's password.
        nope = requests.put(f"{V4}/users/{ADMIN_USER_ID}/password", headers=as_member,
                            timeout=30, json={"new_password": "nope"})
        record("3.1", "non-admin password reset -> 403", nope.status_code == 403,
               f"{nope.status_code} {err(nope).get('code')}")

    finally:
        # ---- teardown, inline ------------------------------------------------
        if lc_group_id is not None and lc_user_id is not None:
            r = requests.delete(f"{V4}/groups/{lc_group_id}/members/{lc_user_id}",
                                headers=V4_ADMIN, timeout=30)
            record("3.1", "DELETE group member -> 204", r.status_code == 204,
                   str(r.status_code))
            if r.status_code == 204:
                untrack("memberships", (lc_group_id, lc_user_id))
                again = requests.delete(f"{V4}/groups/{lc_group_id}/members/{lc_user_id}",
                                        headers=V4_ADMIN, timeout=30)
                record("3.1", "removing a membership twice is idempotent (204)",
                       again.status_code == 204, str(again.status_code))

        if lc_group_id is not None:
            r = requests.delete(f"{V4}/groups/{lc_group_id}", headers=V4_ADMIN, timeout=30)
            record("3.1", "DELETE /v4/groups/{id} -> 204", r.status_code == 204,
                   f"{r.status_code} {err(r).get('code', '')}")
            if r.status_code == 204:
                untrack("groups", lc_group_id)

        if lc_user_id is not None:
            r = requests.delete(f"{V4}/users/{lc_user_id}", headers=V4_ADMIN, timeout=30)
            record("3.1", "DELETE /v4/users/{id} -> 204", r.status_code == 204,
                   f"{r.status_code} {err(r).get('code', '')}")
            if r.status_code == 204:
                untrack("users", lc_user_id)

### 3.2 Reference lists

Small, and read-only, so these run whatever the switches say. They are what a form needs
before it can be filled in: the ISO 639-3 language codes and the ISO 15924 script codes.

In [ ]:
for path, key, probe in (("languages", "iso639", ISO_LANGUAGE),
                         ("scripts", "iso15924", ISO_SCRIPT)):
    page = check(requests.get(f"{V4}/{path}", headers=V4_ADMIN, timeout=60),
                 expect=200).json()
    record("3.2", f"GET /v4/{path} uses the V4Page envelope",
           set(page) == V4_PAGE_KEYS, f"total={page['total']}")
    record("3.2", f"/v4/{path} rows are {{{key}, name}}",
           bool(page["items"]) and set(page["items"][0]) == {key, "name"},
           str(page["items"][0]) if page["items"] else "empty")

    # The `q` filter is what makes the list usable at this size.
    filtered = check(requests.get(f"{V4}/{path}", params={"q": probe}, headers=V4_ADMIN,
                                  timeout=60)).json()
    hit = next((row for row in filtered["items"] if row[key] == probe), None)
    record("3.2", f"/v4/{path}?q={probe} finds {probe}", hit is not None, str(hit))
    record("3.2", f"/v4/{path}?q= narrows the result set",
           filtered["total"] < page["total"],
           f"{filtered['total']} of {page['total']}")

### 3.3 Predictions

Comparing two pieces of text without setting up a whole assessment first.

**The fan-out is pinned to the cheap analyses here, deliberately.** Left to its defaults
`POST /v4/predictions` fans out to all six apps with `include_translation` and
`include_critique` both on, which spawns a background LLM agent pass — and there is no
`DELETE` for a prediction job, so that row could not be cleaned up. With the agent's slow
legs off, `job` comes back `null` and nothing persists.

`POST /v4/predictions/semantic-similarity` needs a **trained** version pair, so against
throwaway fixtures it is expected to refuse. The refusal is the check: it must arrive as a
proper v4 error envelope and not a 500. (This is also the endpoint whose v3 twin has
returned 503 since April 2026 — the runner renamed its entry point and frozen v3 never
followed. v4 calls the current name.)

In [ ]:
# Length comparison is pure arithmetic over two strings — no model, no revision, no cost.
source_text = "In the beginning God created the heavens and the earth."
target_text = "Hapo mwanzo Mungu aliumba mbingu na nchi."

r = check(requests.post(f"{V4}/predictions/length-comparison", headers=V4_ADMIN, timeout=60,
                        json={"source_text": source_text, "target_text": target_text}),
          expect=200)
lc = r.json()
print(json.dumps(lc, indent=2))
record("3.3", "POST /v4/predictions/length-comparison -> 200",
       set(lc) == {"word_count_difference", "char_count_difference"}, str(sorted(lc)))
record("3.3", "word_count_difference is the real difference",
       lc["word_count_difference"] == len(source_text.split()) - len(target_text.split()),
       str(lc["word_count_difference"]))

In [ ]:
if not RUN_PREDICT_FANOUT:
    print("skipped — RUN_PREDICT_FANOUT is False (§0.5)")
else:
    # text-lengths needs no trained model and no selectors; the agent is left out, so
    # no background job is spawned and nothing is written.
    body = {
        "pairs": [{"vref": "GEN 1:1", "source_text": source_text, "target_text": target_text}],
        "apps": ["text-lengths"],
        "include_translation": False,
        "include_critique": False,
    }
    r = check(requests.post(f"{V4}/predictions", headers=V4_ADMIN, json=body, timeout=180),
              expect=200)
    out = r.json()
    print(json.dumps(out, indent=2)[:1200])

    record("3.3", "POST /v4/predictions -> 200 with {pairs, results, job}",
           set(out) == {"pairs", "results", "job"}, str(sorted(out)))
    record("3.3", "no agent pass requested -> no job row created", out["job"] is None,
           str(out["job"]))
    record("3.3", "the request's pairs are echoed back",
           len(out["pairs"]) == len(body["pairs"]))

    for app, entry in out["results"].items():
        record("3.3", f"fan-out entry '{app}' carries the per-app envelope",
               set(entry) == {"status", "data", "error", "duration_ms"},
               f"status={entry.get('status')} {entry.get('duration_ms')}ms")
        record("3.3", f"fan-out entry '{app}' reports a known status",
               entry.get("status") in {"ok", "error", "not_trained"},
               str(entry.get("status")))

In [ ]:
# Semantic similarity against the throwaway pair. A trained model is what it needs, and
# brand-new fixture versions have none — so a refusal is the expected answer here, and
# what is being checked is that it refuses *properly*.
if not RUN_WRITE_TESTS or V3_VERSION_ID is None or V4_VERSION_ID is None:
    print("skipped — needs RUN_WRITE_TESTS and the §2.1 versions")
else:
    r = requests.post(f"{V4}/predictions/semantic-similarity", headers=V4_ADMIN, timeout=180,
                      json={"source_text": source_text, "target_text": target_text,
                            "source_version_id": V3_VERSION_ID,
                            "target_version_id": V4_VERSION_ID})
    print(r.status_code, r.text[:400])
    if r.status_code == 200:
        record("3.3", "semantic-similarity returned a score",
               set(r.json()) == {"score"}, str(r.json()))
    else:
        record("3.3", "untrained pair refused with a v4 error envelope, not a 500",
               r.status_code in (422, 503) and bool(err(r).get("code")),
               f"{r.status_code} {err(r).get('code')}")

In [ ]:
# The poll endpoint, checked on the one path that needs no job: an id that is not one.
r = requests.get(f"{V4}/predictions/not-a-real-job-id", headers=V4_ADMIN, timeout=30)
record("3.3", "GET /v4/predictions/{unknown} -> 404 with a code",
       r.status_code == 404 and bool(err(r).get("code")),
       f"{r.status_code} {err(r).get('code')}")

### 3.4 Training

Reads first, and those always run — they are what a promotion pass actually needs to know
works. `GET /v4/training-jobs` is the entry point; a job carries the session it belongs
to, and the session is what you poll.

The **submit** is gated on `RUN_TRAINING` *and* `RUN_WRITE_TESTS`, because it is real GPU
work and because `DELETE /v4/training-jobs/{id}` is a 409 until the job is terminal. If
the poll below times out, §6 cannot delete the job and will name it for you to clean up
by hand later.

In [ ]:
# Not check(): a 500 here is precisely what this notebook exists to surface, and a
# raising helper would end the cell and hide every check below it.
jobs_resp = requests.get(f"{V4}/training-jobs", params={"limit": 20}, headers=V4_ADMIN,
                         timeout=60)
jobs_page = jobs_resp.json() if jobs_resp.status_code == 200 else None
record("3.4", "GET /v4/training-jobs (unfiltered) -> 200", jobs_page is not None,
       f"{jobs_resp.status_code} {err(jobs_resp).get('code', '')} {jobs_resp.text[:160]}")

sample_job = None
if jobs_page is not None:
    record("3.4", "GET /v4/training-jobs uses the V4Page envelope",
           set(jobs_page) == V4_PAGE_KEYS, f"total={jobs_page['total']}")
    sample_job = jobs_page["items"][0] if jobs_page["items"] else None
else:
    # The unfiltered read failed. Narrowing by type is worth trying anyway: if the
    # filtered reads succeed, the fault is a row the unfiltered query includes and the
    # filter excludes — a stored `type` outside the TrainingType enum, say — rather
    # than the endpoint being down.
    print("  unfiltered read failed; retrying narrowed by type to find a usable job")
    for training_type in ("ngrams", "tfidf", "word-alignment", "semantic-similarity",
                          "agent-critique"):
        probe = requests.get(f"{V4}/training-jobs",
                             params={"type": training_type, "limit": 5},
                             headers=V4_ADMIN, timeout=60)
        print(f"    ?type={training_type:20} -> {probe.status_code}")
        if probe.status_code == 200 and probe.json()["items"] and sample_job is None:
            sample_job = probe.json()["items"][0]
    record("3.4", "filtered training-jobs reads still work", sample_job is not None,
           "the unfiltered query is what breaks, not the endpoint"
           if sample_job else "narrowed reads failed too")

if sample_job is None:
    record("3.4", "training job reads", True, "skipped — no usable training job found")
else:
    print(json.dumps(sample_job, indent=2)[:800])
    record("3.4", "training job reports a closed JobState (or null)",
           sample_job.get("state") in {None, "PENDING", "RUNNING", "SUCCEEDED", "FAILED"},
           str(sample_job.get("state")))

    one = requests.get(f"{V4}/training-jobs/{sample_job['id']}", headers=V4_ADMIN, timeout=30)
    record("3.4", "GET /v4/training-jobs/{id} answers 200 or 202",
           one.status_code in (200, 202), str(one.status_code))
    if one.ok:
        record("3.4", "job detail carries the job envelope",
               {"job_id", "state", "result", "error"} <= set(one.json()),
               str(one.json().get("state")))

    # Filters must actually filter.
    typed = check(requests.get(f"{V4}/training-jobs",
                               params={"type": sample_job["type"], "limit": 5},
                               headers=V4_ADMIN, timeout=60)).json()
    record("3.4", f"?type={sample_job['type']} filters the list",
           all(j["type"] == sample_job["type"] for j in typed["items"]),
           f"{typed['total']} of "
           f"{jobs_page['total'] if jobs_page is not None else 'unknown'}")

    # Follow a job to its session, then read that session's results.
    session_id = sample_job.get("session_id")
    if not session_id:
        record("3.4", "session read", True, "skipped — the sampled job has no session_id")
    else:
        s = check(requests.get(f"{V4}/training-sessions/{session_id}", headers=V4_ADMIN,
                               timeout=60), expect=200).json()
        record("3.4", "GET /v4/training-sessions/{id} is {session_id, state, jobs, inference_readiness}",
               set(s) == {"session_id", "state", "jobs", "inference_readiness"},
               str(sorted(s)))
        record("3.4", "the session lists the job we came from",
               sample_job["id"] in [j["id"] for j in s["jobs"]])

        res = requests.get(f"{V4}/training-sessions/{session_id}/results",
                           params={"limit": 5}, headers=V4_ADMIN, timeout=120)
        # The handler has exactly two outcomes of its own: 200, or 404 for a session
        # it cannot resolve. Anything else — a 500 especially — is the regression.
        record("3.4", "GET /v4/training-sessions/{id}/results answers to contract",
               res.status_code == 200 or
               (res.status_code == 404 and bool(err(res).get("code"))),
               f"{res.status_code} {err(res).get('code', '')}")
        if res.status_code == 200:
            page = res.json()
            record("3.4", "results use the V4Page envelope",
                   set(page) == V4_PAGE_KEYS, f"total={page['total']}")
            if page["items"]:
                print(json.dumps(page["items"][0], indent=2)[:600])

In [ ]:
# ---- the submit. Off by default; see §0.5 and the note above. ----------------
TRAINING_POLL_TIMEOUT_S = 900   # 15 minutes
TRAINING_POLL_INTERVAL_S = 20

if not (RUN_WRITE_TESTS and RUN_TRAINING):
    print("skipped — needs RUN_WRITE_TESTS *and* RUN_TRAINING (§0.5)")
elif V3_REVISION_ID is None or V4_REVISION_ID is None:
    print("skipped — needs both §2.2 revisions as the pair to train on")
else:
    # One app, the cheapest, rather than the default (which is all five and includes
    # agent-critique).
    r = requests.post(f"{V4}/training-sessions", headers=V4_ADMIN, timeout=120, json={
        "source_revision_id": V3_REVISION_ID,
        "target_revision_id": V4_REVISION_ID,
        "apps": ["ngrams"],
    })
    if r.status_code == 409:
        record("3.4", "training submit", True,
               f"409 TRAINING_JOBS_ALREADY_ACTIVE — {err(r).get('details')}")
    else:
        check(r, expect=202)
        session_id = r.json()["job_id"]
        record("3.4", "POST /v4/training-sessions -> 202", True, f"session {session_id}")
        record("3.4", "202 carries Location and Retry-After",
               "location" in {k.lower() for k in r.headers} and
               "retry-after" in {k.lower() for k in r.headers},
               f"Location={r.headers.get('Location')} Retry-After={r.headers.get('Retry-After')}")

        deadline = time.time() + TRAINING_POLL_TIMEOUT_S
        session = None
        while time.time() < deadline:
            session = check(requests.get(f"{V4}/training-sessions/{session_id}",
                                         headers=V4_ADMIN, timeout=60)).json()
            for job in session["jobs"]:
                track("training_jobs", job["id"])
            print(f"  {session['state']}  " +
                  ", ".join(f"{j['type']}={j['state']}" for j in session["jobs"]))
            if session["state"] in ("SUCCEEDED", "FAILED"):
                break
            time.sleep(TRAINING_POLL_INTERVAL_S)

        terminal = session is not None and session["state"] in ("SUCCEEDED", "FAILED")
        record("3.4", "the training session reached a terminal state", terminal,
               str(session["state"]) if session else "no poll succeeded")
        if not terminal:
            print("\n!! Still running. §6 will try to delete it and report a 409 if it "
                  "cannot — delete these job ids by hand once they finish:")
            print("   ", CREATED["training_jobs"])

### 3.5 Agent critique

Reads over what the agent found and the text it produced. These need an existing
agent-critique assessment with data, which the cell discovers by walking back from the
most recent run — creating one is not an option here: it is LLM work, the runner caps it
at a single chapter per run, and the fixture revision is three lines of filler. If nothing suitable is visible to this account,
the cell says so and skips rather than fabricating anything.

The `PATCH` is the one write in this section, and it writes to a **real critique issue on
the shared database**. It therefore captures the row's original `resolved` and
`resolution_notes` first and restores them in a `finally`, so the net change is nothing.
It is gated on `RUN_WRITE_TESTS` like every other write.

In [ ]:
CRITIQUE_ASSESSMENT_ID = None
CRITIQUE_ISSUE = None

# The list is ordered by id ascending, so the FIRST page is the oldest runs — which
# predate critique-issue storage and have none. Searching from that end finds nothing
# and reports "no data" for a surface that is fine. Jump to the last page instead and
# work backwards from the newest.
head = check(requests.get(f"{V4}/assessments",
                          params={"type": "agent-critique", "limit": 1},
                          headers=V4_ADMIN, timeout=60), expect=200).json()
total_critiques = head["total"]
page = check(requests.get(f"{V4}/assessments",
                          params={"type": "agent-critique", "limit": 100,
                                  "offset": max(0, total_critiques - 100)},
                          headers=V4_ADMIN, timeout=60)).json()
candidates = [a for a in reversed(page["items"]) if a["state"] == "SUCCEEDED"]
print(f"{total_critiques} agent-critique assessments visible; probing the "
      f"{len(candidates)} most recent that succeeded, newest first")

# Only *unresolved* issues are eligible, and that is a correctness requirement, not
# tidiness. Re-asserting an existing resolution as a different account is not a no-op:
# resolve_critique_issue writes whenever the asserting user differs, taking over
# `resolved_by_id` and `resolved_at`. So "flip it and put it back" on an already-resolved
# issue would permanently reassign a real row's resolution to this admin account.
# Starting from unresolved makes the restore genuinely empty.
for candidate in candidates[:15]:
    probe = requests.get(f"{V4}/assessments/{candidate['id']}/critique-issues",
                         params={"limit": 1, "resolved": False}, headers=V4_ADMIN,
                         timeout=60)
    if probe.ok and probe.json()["total"]:
        CRITIQUE_ASSESSMENT_ID = candidate["id"]
        CRITIQUE_ISSUE = probe.json()["items"][0]
        break

if CRITIQUE_ASSESSMENT_ID is None:
    record("3.5", "agent critique reads", True,
           "skipped — no visible agent-critique assessment has an unresolved issue")
else:
    print(f"using assessment {CRITIQUE_ASSESSMENT_ID}, issue {CRITIQUE_ISSUE['id']}")
    print(json.dumps(CRITIQUE_ISSUE, indent=2)[:900])

    issues = check(requests.get(f"{V4}/assessments/{CRITIQUE_ASSESSMENT_ID}/critique-issues",
                                params={"limit": 20}, headers=V4_ADMIN, timeout=60)).json()
    record("3.5", "GET critique-issues uses the V4Page envelope",
           set(issues) == V4_PAGE_KEYS, f"total={issues['total']}")
    # Bible order is *canonical* book order, which is not alphabetical — so sorting
    # by the book name would be checking the wrong thing. Within a single book the
    # order is just (chapter, verse), and a critique run covers one chapter, so that
    # is the case worth asserting; across books, say so and check nothing.
    books = {i["book"] for i in issues["items"]}
    if len(issues["items"]) < 2:
        record("3.5", "issues come back in Bible order", True,
               "not checked — fewer than two rows")
    elif len(books) == 1:
        order = [(i["chapter"], i["verse"]) for i in issues["items"]]
        record("3.5", "issues come back in Bible order", order == sorted(order),
               f"{len(order)} rows in {next(iter(books))}")
    else:
        record("3.5", "issues come back in Bible order", True,
               f"not checked — spans {len(books)} books and canonical order is not "
               f"alphabetical")

    unresolved = check(requests.get(
        f"{V4}/assessments/{CRITIQUE_ASSESSMENT_ID}/critique-issues",
        params={"resolved": False, "limit": 20}, headers=V4_ADMIN, timeout=60)).json()
    record("3.5", "?resolved=false returns only unresolved issues",
           all(i["resolved"] is False for i in unresolved["items"]),
           f"{len(unresolved['items'])} of {unresolved['total']}")

    translations = check(requests.get(f"{V4}/assessments/{CRITIQUE_ASSESSMENT_ID}/translations",
                                      params={"limit": 5}, headers=V4_ADMIN, timeout=60)).json()
    record("3.5", "GET translations uses the V4Page envelope",
           set(translations) == V4_PAGE_KEYS,
           f"total={translations['total']}")
    if translations["items"]:
        record("3.5", "a translation row carries its vref and attempt",
               {"vref", "vrefs", "attempt", "assessment_id"} <= set(translations["items"][0]),
               str(sorted(translations["items"][0])[:8]))

In [ ]:
# PATCH resolve / reopen, then put the row back exactly as it was.
if not RUN_WRITE_TESTS:
    print("skipped — RUN_WRITE_TESTS is False (§0.5)")
elif CRITIQUE_ISSUE is None:
    print("skipped — §3.5 found no critique issue to work on")
elif CRITIQUE_ISSUE["resolved"]:
    # Belt and braces: §3.5's search already filters to resolved=false.
    print("skipped — the discovered issue is already resolved, and re-asserting that "
          "resolution as this account would take over resolved_by_id and resolved_at "
          "on a real row rather than restoring it.")
else:
    issue_id = CRITIQUE_ISSUE["id"]
    url = f"{V4}/assessments/{CRITIQUE_ASSESSMENT_ID}/critique-issues/{issue_id}"
    note = f"{PREFIX} pre-promotion check"
    print(f"issue {issue_id} starts unresolved; resolving then reopening it")

    try:
        # Resolve. Notes are accepted only alongside resolved: true — the body
        # validator refuses them with resolved: false, since unresolving clears them.
        body = check(requests.patch(url, headers=V4_ADMIN, timeout=30,
                                    json={"resolved": True, "resolution_notes": note}),
                     expect=200).json()
        record("3.5", "PATCH critique issue -> resolved=True",
               body["resolved"] is True, str(body["resolved"]))
        record("3.5", "PATCH echoes the note it was given",
               body.get("resolution_notes") == note, str(body.get("resolution_notes")))
        record("3.5", "resolving stamps resolved_by_id and resolved_at",
               body.get("resolved_by_id") == ADMIN_USER_ID and bool(body.get("resolved_at")),
               f"by={body.get('resolved_by_id')} at={body.get('resolved_at')}")

        # Reopen — the other half of the same endpoint. No notes: resolved: false
        # clears all four fields together, and sending notes here is a 422.
        body = check(requests.patch(url, headers=V4_ADMIN, timeout=30,
                                    json={"resolved": False}), expect=200).json()
        record("3.5", "PATCH back to resolved=False", body["resolved"] is False,
               str(body["resolved"]))
        record("3.5", "reopening clears the resolution fields",
               body.get("resolution_notes") is None and body.get("resolved_by_id") is None
               and body.get("resolved_at") is None,
               f"notes={body.get('resolution_notes')} by={body.get('resolved_by_id')}")

        # Sending notes with resolved: false is refused, not silently dropped.
        bad = requests.patch(url, headers=V4_ADMIN, timeout=30,
                             json={"resolved": False, "resolution_notes": note})
        record("3.5", "resolution_notes with resolved=false -> 422",
               bad.status_code == 422 and err(bad).get("code") == "VALIDATION_ERROR",
               f"{bad.status_code} {err(bad).get('code')}")

    finally:
        # Put the row back where it started: unresolved, no notes. Asserting the state
        # that already holds issues no UPDATE at all, so this is a true no-op whenever
        # the body above already succeeded.
        restore = requests.patch(url, headers=V4_ADMIN, timeout=30,
                                 json={"resolved": False})
        ok = restore.status_code == 200 and restore.json()["resolved"] is False
        record("3.5", "issue left unresolved, as found", ok,
               f"{restore.status_code}")
        if not ok:
            print(f"!! issue {issue_id} may still be resolved — check it by hand")

## 4. Error contract

v4 answers every error with `{"error": {"code", "message", "details"}}` and clients branch
on the stable `code`. v3 answers with a flat `{"detail": ...}`. That divergence is
deliberate, and it is the contract change a client actually has to code against — so it is
worth proving on a live deployment rather than in a unit test.

The probes below either send a body that cannot validate or ask for an id that does not
exist, so none of them writes anything. They run whatever the switches say.

The `extra=forbid` check is the one that matters most for this promotion. Every v4 request
body rejects unrecognised fields, which is what turns the #925 renames from a silent data
loss into a clear 422 — a client still sending `forward_translation` must be *told*, not
quietly ignored.

In [ ]:
# --- a required field is missing -------------------------------------------
r = requests.post(f"{V4}/versions", headers=V4_ADMIN, timeout=30, json={
    "name": f"{PREFIX}-never-created",
    "iso_language": ISO_LANGUAGE,
    "iso_script": ISO_SCRIPT,
    # abbreviation and group_ids both missing
})
print(json.dumps(r.json(), indent=2)[:700])
record("4", "missing required field -> 422 VALIDATION_ERROR",
       r.status_code == 422 and err(r).get("code") == "VALIDATION_ERROR",
       f"{r.status_code} {err(r).get('code')}")
record("4", "422 envelope is {code, message, details}",
       set(err(r)) == {"code", "message", "details"}, str(sorted(err(r))))
record("4", "details name the fields that failed",
       bool(err(r).get("details")), str(err(r).get("details"))[:200])

In [ ]:
# --- an unrecognised field (extra=forbid) ------------------------------------
# `group_ids: []` makes this doubly safe: even if extra=forbid had regressed, the
# service rejects an empty group list before writing anything.
r = requests.post(f"{V4}/versions", headers=V4_ADMIN, timeout=30, json={
    "name": f"{PREFIX}-never-created",
    "iso_language": ISO_LANGUAGE,
    "iso_script": ISO_SCRIPT,
    "abbreviation": "zzz",
    "group_ids": [],
    "forward_translation": 1,   # the pre-#925 spelling; must not be silently dropped
})
print(r.status_code, json.dumps(r.json(), indent=2)[:700])

record("4", "unknown field -> 422 (v4 bodies are closed)",
       r.status_code == 422 and err(r).get("code") == "VALIDATION_ERROR",
       f"{r.status_code} {err(r).get('code')}")
record("4", "the 422 names the offending field rather than dropping it",
       "forward_translation" in json.dumps(err(r).get("details")),
       str(err(r).get("details"))[:250])

if r.status_code == 201:
    # Should be unreachable; track it so §6 cleans up after the regression.
    track("versions", r.json()["id"])
    print("!! extra=forbid has regressed — a version was created and has been tracked "
          "for teardown")

In [ ]:
# --- an id that does not exist ------------------------------------------------
for path, expected_code in (("versions/999999999", "VERSION_NOT_FOUND"),
                            ("revisions/999999999", "REVISION_NOT_FOUND"),
                            ("assessments/999999999", "ASSESSMENT_NOT_FOUND")):
    r = requests.get(f"{V4}/{path}", headers=V4_ADMIN, timeout=30)
    record("4", f"GET /v4/{path} -> 404 {expected_code}",
           r.status_code == 404 and err(r).get("code") == expected_code,
           f"{r.status_code} {err(r).get('code')}")

# --- pagination bounds are rejected, never clamped ----------------------------
for params, why in (({"limit": 0}, "limit below the floor"),
                    ({"limit": 101}, "limit above MAX_LIMIT"),
                    ({"offset": -1}, "negative offset")):
    r = requests.get(f"{V4}/versions", params=params, headers=V4_ADMIN, timeout=30)
    record("4", f"{why} -> 422 VALIDATION_ERROR",
           r.status_code == 422 and err(r).get("code") == "VALIDATION_ERROR",
           f"{r.status_code} {err(r).get('code')}")

# --- no token -----------------------------------------------------------------
r = requests.get(f"{V4}/versions", timeout=30)
record("4", "no token -> 401 (router-level auth is on)", r.status_code == 401,
       f"{r.status_code} {err(r).get('code')}")

r = requests.post(f"{V4}/token", data={"username": "admin", "password": "not-the-password"},
                  timeout=30)
record("4", "bad credentials -> 401 INVALID_CREDENTIALS",
       r.status_code == 401 and err(r).get("code") == "INVALID_CREDENTIALS",
       f"{r.status_code} {err(r).get('code')}")
record("4", "v3's flat {'detail': ...} is gone from v4",
       "detail" not in r.json(), str(sorted(r.json())))

In [ ]:
# --- and the v3 side is unchanged ---------------------------------------------
# v3 keeps its flat body. If this starts answering with an envelope, something has
# leaked across the mount boundary.
r = requests.get(f"{V3}/version", timeout=30)
record("4", "v3 still answers 401 with a flat {'detail': ...}",
       r.status_code == 401 and "detail" in r.json() and "error" not in r.json(),
       f"{r.status_code} {str(r.json())[:80]}")

## 5. v3 regression spot-checks

The promotion PR claims "no v3 endpoint is added, removed, or renamed" and "nothing in v3
is affected". §1 checked the operation count; this section checks the behaviour, by
re-running `integration.ipynb`'s own lifecycle against the promoted `main`: create a
version, upload a revision, run the three assessment types it covers plus ngrams, read the
results back, delete everything.

This is deliberately a copy of what the existing notebook does rather than an improvement
on it — the value is that it is the same path the website walks.

Note what moved underneath it in this promotion: FastAPI 0.115 → 0.141, Starlette 0.41 →
1.6, Pydantic 2.4 → 2.9, and python-jose → PyJWT. Nothing here should notice.

In [ ]:
V3_REG_VERSION_ID = None
V3_REG_REVISION_ID = None
V3_REG_ASSESSMENT_IDS = []

if not RUN_WRITE_TESTS or GROUP_ID is None:
    print("skipped — needs RUN_WRITE_TESTS and the §2.0 group")
else:
    new_version_data = {
        "name": f"{PREFIX}-v3-regression",
        "iso_language": ISO_LANGUAGE,
        "iso_script": ISO_SCRIPT,
        "abbreviation": f"{PREFIX[:12]}R",
        "rights": "Some Rights",
        "machineTranslation": False,
        "is_reference": True,
        "add_to_groups": [GROUP_ID],
    }
    version = check(requests.post(f"{V3}/version", json=new_version_data,
                                  headers=V3_ADMIN, timeout=60)).json()
    V3_REG_VERSION_ID = track("versions", version["id"])
    track("version_grants", (V3_REG_VERSION_ID, GROUP_ID))
    record("5", "v3 version create", True, f"version {V3_REG_VERSION_ID}")

    # PUT /version — the rename path integration.ipynb exercises.
    check(requests.put(f"{V3}/version",
                       json={"id": V3_REG_VERSION_ID, "name": f"{PREFIX}-renamed"},
                       headers=V3_ADMIN, timeout=60))
    after = next(v for v in check(requests.get(f"{V3}/version", headers=V3_ADMIN,
                                               timeout=60)).json()
                 if v["id"] == V3_REG_VERSION_ID)
    record("5", "v3 version rename took effect", after["name"] == f"{PREFIX}-renamed",
           after["name"])

    with open(UPLOAD_FIXTURE, "rb") as fh:
        revision = check(requests.post(
            f"{V3}/revision",
            params={"version_id": V3_REG_VERSION_ID, "name": f"{PREFIX}-v3-reg-rev"},
            files={"file": fh}, headers=V3_ADMIN, timeout=180,
        )).json()
    V3_REG_REVISION_ID = track("revisions", revision["id"])
    record("5", "v3 revision upload", True, f"revision {V3_REG_REVISION_ID}")

    listed = check(requests.get(f"{V3}/revision", params={"version_id": V3_REG_VERSION_ID},
                                headers=V3_ADMIN, timeout=60)).json()
    record("5", "the uploaded revision appears in GET /latest/revision",
           V3_REG_REVISION_ID in [x["id"] for x in listed], f"{len(listed)} rows")

In [ ]:
if not RUN_WRITE_TESTS or V3_REG_REVISION_ID is None:
    print("skipped — needs the §5 revision")
else:
    # The three integration.ipynb covers, plus ngrams — same calls, same order.
    for assessment_type in ("semantic-similarity", "word-alignment", "sentence-length",
                            "ngrams"):
        r = requests.post(f"{V3}/assessment",
                          params={"revision_id": V3_REG_REVISION_ID,
                                  "reference_id": V3_REG_REVISION_ID,
                                  "type": assessment_type},
                          headers=V3_ADMIN, timeout=180)
        ok = r.ok and isinstance(r.json(), list) and r.json()
        record("5", f"v3 assessment create: {assessment_type}", bool(ok),
               f"{r.status_code} {r.text[:120]}")
        if ok:
            aid = track("assessments", r.json()[0]["id"])
            V3_REG_ASSESSMENT_IDS.append(aid)
            print(f"  {assessment_type}: assessment {aid} status={r.json()[0].get('status')}")

In [ ]:
if not V3_REG_ASSESSMENT_IDS:
    print("skipped — no §5 assessments were created")
else:
    for aid in V3_REG_ASSESSMENT_IDS:
        r = requests.get(f"{V3}/result", params={"assessment_id": aid},
                         headers=V3_ADMIN, timeout=120)
        ok = r.ok and set(r.json()) == {"results", "total_count"}
        record("5", f"v3 GET /latest/result for assessment {aid}", ok,
               f"{r.status_code} total_count={r.json().get('total_count') if r.ok else '-'}")

## 6. Teardown

**Run this cell even if something above failed.** It walks the registry every create cell
appended to, in dependency order:

1. assessments (soft delete)
2. revisions (soft delete)
3. version → group access grants, revoked **before** the groups are deleted: a soft-deleted
   version still holds its access rows, and `DELETE /v4/groups/{id}` refuses with a 409
   while any grant survives
4. versions (soft delete)
5. group memberships
6. groups (hard delete)
7. users (hard delete)
8. training jobs — **terminal ones only**; a still-running job is a 409 and is reported
   rather than retried

Every delete here is idempotent, so re-running this cell is safe and re-running the whole
notebook does not accumulate junk.

In [ ]:
def teardown():
    leftovers = []

    def attempt(label, response, drop):
        ok = response.status_code in (200, 204)
        record("6", label, ok, f"{response.status_code} {err(response).get('code', '')}")
        if ok:
            drop()
        else:
            leftovers.append(f"{label} -> {response.status_code} {response.text[:120]}")
        return ok

    for aid in list(CREATED["assessments"]):
        attempt(f"delete assessment {aid}",
                requests.delete(f"{V4}/assessments/{aid}", headers=V4_ADMIN, timeout=60),
                lambda aid=aid: untrack("assessments", aid))

    for rid in list(CREATED["revisions"]):
        attempt(f"delete revision {rid}",
                requests.delete(f"{V4}/revisions/{rid}", headers=V4_ADMIN, timeout=120),
                lambda rid=rid: untrack("revisions", rid))

    for version_id, group_id in list(CREATED["version_grants"]):
        attempt(f"revoke version {version_id} access for group {group_id}",
                requests.delete(f"{V4}/versions/{version_id}/groups/{group_id}",
                                headers=V4_ADMIN, timeout=60),
                lambda k=(version_id, group_id): untrack("version_grants", k))

    for vid in list(CREATED["versions"]):
        attempt(f"delete version {vid}",
                requests.delete(f"{V4}/versions/{vid}", headers=V4_ADMIN, timeout=120),
                lambda vid=vid: untrack("versions", vid))

    for group_id, user_id in list(CREATED["memberships"]):
        attempt(f"remove user {user_id} from group {group_id}",
                requests.delete(f"{V4}/groups/{group_id}/members/{user_id}",
                                headers=V4_ADMIN, timeout=60),
                lambda k=(group_id, user_id): untrack("memberships", k))

    for gid in list(CREATED["groups"]):
        attempt(f"delete group {gid}",
                requests.delete(f"{V4}/groups/{gid}", headers=V4_ADMIN, timeout=60),
                lambda gid=gid: untrack("groups", gid))

    for uid in list(CREATED["users"]):
        attempt(f"delete user {uid}",
                requests.delete(f"{V4}/users/{uid}", headers=V4_ADMIN, timeout=60),
                lambda uid=uid: untrack("users", uid))

    # Training jobs last: a 409 here means "still running", which is expected and not
    # something to retry. It is reported so you can come back to it.
    for jid in list(CREATED["training_jobs"]):
        r = requests.delete(f"{V4}/training-jobs/{jid}", headers=V4_ADMIN, timeout=60)
        if r.status_code == 204:
            record("6", f"delete training job {jid}", True, "204")
            untrack("training_jobs", jid)
        else:
            record("6", f"delete training job {jid}", False,
                   f"{r.status_code} {err(r).get('code', '')} — delete by hand once terminal")
            leftovers.append(f"training job {jid} -> {r.status_code} {err(r).get('code', '')}")

    return leftovers


leftovers = teardown()

print("\n" + "=" * 70)
remaining = {k: v for k, v in CREATED.items() if v}
if remaining:
    print(f"NOT CLEANED UP (prefix {PREFIX}):")
    print(json.dumps(remaining, indent=2))
    for line in leftovers:
        print("  ", line)
else:
    print(f"Clean — nothing created by this run ({PREFIX}) is left behind.")

## 7. Summary

In [ ]:
width = max((len(label) for _, label, _, _ in RESULTS), default=10)
passed = sum(1 for _, _, ok, _ in RESULTS if ok)

print(f"target : {base_url}")
print(f"prefix : {PREFIX}")
print(f"writes : {'ENABLED' if RUN_WRITE_TESTS else 'disabled'}"
      f"   training: {'ENABLED' if (RUN_WRITE_TESTS and RUN_TRAINING) else 'disabled'}")
print()
print(f"{'§':<5} {'check':<{width}}  result")
print("-" * (width + 22))
for section, label, ok, detail in RESULTS:
    print(f"{section:<5} {label:<{width}}  {'PASS' if ok else 'FAIL'}"
          + (f"  ({detail})" if detail and not ok else ""))
print("-" * (width + 22))
print(f"{passed}/{len(RESULTS)} passed")

failures = [(s, l, d) for s, l, ok, d in RESULTS if not ok]
if failures:
    print("\nFAILURES")
    for s, l, d in failures:
        print(f"  §{s} {l}" + (f" — {d}" if d else ""))
    print("\nDo not promote until each of these is understood.")
else:
    print("\nv3 and v4 agree on every row checked, and the v4 contract held. "
          "Safe to promote as far as this notebook can tell.")

---

### What this notebook deliberately does not cover

- **The v4 pagination and versions contract in depth** — `v4_smoke.ipynb` already pins the
  `V4Page` envelope, the `users/me` allowlist (#859), out-of-range paging and the versions
  write path. No point owning it twice.
- **The assessment sub-resource reads** (`ngrams`, `similar-verses`, `alignment-scores`,
  `missing-words`, `text-lengths`, `score-comparison`) — they were promoted before this
  round and need assessments with real finished data, which the throwaway fixture does not
  have. Point them at a known-good assessment by hand if a promotion touches them.
- **Creating an agent-critique assessment.** The runner caps a run at one chapter and it is
  LLM work; §3.5 reads what already exists instead.
- **The `Idempotency-Key` header.** Accepted nowhere yet.
- **Prod.** §0.1 has the URL commented out. Pointed there before a promotion, §1 reports
  the *old* operation count, which is the useful before-reading — but every write section
  should stay off.

### When a parity check fails

**Run §6 (Teardown) by hand.** A §2 mismatch raises, and a raise inside "Run All" stops
the queue — every later cell, teardown included, is skipped. That is the one case where
fixtures are left on the shared database, and it is exactly the case this notebook exists
to produce. Teardown is safe to run on its own at any time: the registry it walks is
module state, so it still holds everything created before the failure, and every delete
in it is idempotent.

The raise itself is deliberate — the prompt for a parity check is that it should be loud.
Read the printed table before re-running: it names every field that differed, on both
surfaces, before it raises, so one mismatched timestamp does not hide a second one two
rows down.